# Publish lecture graphics

Regenerates every SVG in `img/` and pushes the changes to GitHub. Runs entirely in Colab.

**One-time setup**

1. Create a fine-grained personal access token at
   https://github.com/settings/personal-access-tokens/new
   - Repository access: only `youngashin.github.io`
   - Permissions: Contents -> **Read and write**
   - Expiration: whatever suits the semester
2. In this notebook, open the key icon on the left sidebar (Secrets).
3. Add a secret named `GITHUB_TOKEN`, paste the token as the value, and switch on
   *Notebook access*.

The token never appears in this notebook and never gets committed. It lives in your
Colab account and is masked out of every log line below.

**Each time you want to publish**

Runtime -> Run all. That is the whole workflow.

---

The lecture notebooks themselves do not need this. Use `File -> Save a copy in GitHub`
from inside the notebook you are editing; Colab commits it directly.

In [ ]:
# @title 1. Connect to the repository
import os, subprocess
from google.colab import userdata

USER   = "youngah-shin"  # @param {type:"string"}
REPO   = "youngashin.github.io"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
NAME   = "youngah-shin"  # @param {type:"string"}
EMAIL  = "youngah2026@iscu.ac.kr"  # @param {type:"string"}

try:
    TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    raise SystemExit("Add a secret named GITHUB_TOKEN first (key icon, left sidebar).")

def run(cmd, cwd=None, quiet=False):
    """Run a command, masking the token out of anything printed."""
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(TOKEN, "***")
    if not quiet and out.strip():
        print(out.strip())
    if r.returncode:
        raise SystemExit(f"failed: {' '.join(cmd[:3])}")
    return r.stdout

URL = f"https://{TOKEN}@github.com/{USER}/{REPO}.git"

if os.path.isdir(REPO):
    run(["git", "-C", REPO, "fetch", "origin"], quiet=True)
    run(["git", "-C", REPO, "reset", "--hard", f"origin/{BRANCH}"])
else:
    run(["git", "clone", "--depth", "1", "-b", BRANCH, URL])

run(["git", "-C", REPO, "config", "user.name", NAME], quiet=True)
run(["git", "-C", REPO, "config", "user.email", EMAIL], quiet=True)
print(f"\nready — {len(os.listdir(REPO + '/img'))} files currently in img/")

In [ ]:
# @title 2. Regenerate every graphic
IMG = f"{REPO}/img"
for script in ["make_titles_svg.py", "make_figures_svg.py", "make_svg.py"]:
    print(f"--- {script}")
    run(["python", f"../tools/{script}"], cwd=IMG)

In [ ]:
# @title 3. Review what changed
diff = run(["git", "-C", REPO, "status", "--porcelain"], quiet=True)
if diff.strip():
    for line in diff.strip().splitlines():
        print(line)
else:
    print("no changes — nothing to publish")

In [ ]:
# @title 4. Commit and push
MESSAGE = "update lecture graphics"  # @param {type:"string"}

if run(["git", "-C", REPO, "status", "--porcelain"], quiet=True).strip():
    run(["git", "-C", REPO, "add", "-A"], quiet=True)
    run(["git", "-C", REPO, "commit", "-m", MESSAGE])
    run(["git", "-C", REPO, "push", "origin", BRANCH])
    print(f"\npushed. raw URLs refresh within a few minutes:")
    print(f"https://raw.githubusercontent.com/{USER}/{REPO}/{BRANCH}/img/")
else:
    print("nothing to commit")

---

## Editing the graphics

Open `tools/make_titles_svg.py` or `tools/make_figures_svg.py` from the file browser on
the left, change the `CONFIG` block at the top, then run cells 2 to 4 again.

Edits made here are lost when the runtime disconnects unless you push them, so run
cell 4 before closing.

## Uploading a screenshot

Drag the file into `youngashin.github.io/img/` in the file browser, then run
cell 4. No regeneration needed.

## If the push is rejected

Someone (or you, from the web) committed since cell 1 ran. Re-run cell 1 to fetch and
reset, then redo your edits and push again.

## If an image still looks stale

`raw.githubusercontent.com` caches for a few minutes. Append a version marker to force a
refresh: `...w01_cover.svg?v=2`